In [82]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [83]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [84]:


queryProduct = """
SELECT 
    [ProductID],
    [Name],
    [ProductNumber],
    [WeightUnitMeasureCode]
      ,[FinishedGoodsFlag]
      ,[Color]
      ,[SafetyStockLevel]
      ,[ReorderPoint]
      ,[StandardCost]
      ,[ListPrice]
      ,[Size]
      ,[Weight]
      ,[DaysToManufacture]
      ,[ProductLine]
      ,[Class]
      ,[Style]
      ,[ProductSubcategoryID]
      ,[ProductModelID]
      ,[SellStartDate]
      ,[SellEndDate]
FROM Production.Product
"""

dimensionProducto = pd.read_sql_query(queryProduct, motorBaseDatos)



queryProductModel = """
SELECT 
  [ProductModelID],
  [Name]
FROM Production.ProductModel
"""

tablaProductModel = pd.read_sql_query(queryProductModel, motorBaseDatos)



queryProductPhoto = """
SELECT 
    [ProductPhotoID],
    [LargePhoto]
FROM Production.ProductPhoto
"""
tablaProductPhoto = pd.read_sql_query(queryProductPhoto, motorBaseDatos)
# tablaProductPhoto



queryProductDescription = """
SELECT 
[ProductDescriptionID]
      ,[Description]
FROM Production.ProductDescription
"""
tablaProductDescription = pd.read_sql_query(queryProductDescription, motorBaseDatos)

queryProductModelProductDescriptionCulture = """
SELECT 
[ProductModelID]
      ,[ProductDescriptionID]
FROM Production.ProductModelProductDescriptionCulture
"""
tablaProductModelProductDescriptionCulture = pd.read_sql_query(queryProductModelProductDescriptionCulture, motorBaseDatos)



queryProductProductPhoto = """
SELECT 
    [ProductID]
      ,[ProductPhotoID]
FROM Production.ProductProductPhoto
"""
tablaProductProductPhoto = pd.read_sql_query(queryProductProductPhoto, motorBaseDatos)

# tablaProductProductPhoto
# dimensionProducto
# tablaProductModel
# dimensionProducto
# tablaProductDescription
# tablaProductModelProductDescriptionCulture

TRANSFORMACION

In [85]:
description = tablaProductModelProductDescriptionCulture.merge(tablaProductDescription, on='ProductDescriptionID')
description

,ProductModelID,ProductDescriptionID,Description
0,1,1199,"Light-weight, wind-resistant, packs to fit int..."
1,1,1467,علب خفيفة الوزن، ومقاومة للريح، تناسب حجم الجيب.
2,1,1589,Sacs légers et résistants au vent ; tiennent d...
3,1,1712,น้ำหนักเบา กันลม ขนาดกะทัดรัดพอดีกระเป๋า
4,1,1838,"קל-משקל, מגן מרוח, מתקפל לגודל המתאים לכיס."
...,...,...,...
757,127,2006,تصميم عريض الوصلات.
758,127,2007,Conception liaison large.
759,127,2008,การออกแบบให้มีจุดเชื่อมกว้าง
760,127,2009,עיצוב רחב-חוליות.


In [86]:
productPhoto = tablaProductProductPhoto.merge(tablaProductPhoto, on='ProductPhotoID')
productPhoto.drop(columns=[
    'ProductPhotoID'
], inplace=True)
productPhoto

,ProductID,LargePhoto
0,1,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
1,2,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
2,3,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
3,4,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
4,316,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
...,...,...
499,995,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
500,996,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
501,997,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...
502,998,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...


In [87]:
dimensionProducto = dimensionProducto.merge(productPhoto, on='ProductID')


dimensionProducto.rename(columns={
    'Name' : ' EnglishProductName',
}, inplace=True)



dimensionProducto
# print(dimensionProducto.columns)


,ProductID,EnglishProductName,ProductNumber,WeightUnitMeasureCode,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,LargePhoto
0,1,Adjustable Race,AR-5381,None,False,None,1000,750,0.0000,0.00,...,NaN,0,None,None,None,NaN,NaN,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
1,2,Bearing Ball,BA-8327,None,False,None,1000,750,0.0000,0.00,...,NaN,0,None,None,None,NaN,NaN,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
2,3,BB Ball Bearing,BE-2349,None,False,None,800,600,0.0000,0.00,...,NaN,1,None,None,None,NaN,NaN,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
3,4,Headset Ball Bearings,BE-2908,None,False,None,800,600,0.0000,0.00,...,NaN,0,None,None,None,NaN,NaN,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
4,316,Blade,BL-2036,None,False,None,800,600,0.0000,0.00,...,NaN,1,None,None,None,NaN,NaN,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,995,ML Bottom Bracket,BB-8107,G,True,None,500,375,44.9506,101.24,...,168.00,1,None,M,None,5.0,96.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
500,996,HL Bottom Bracket,BB-9108,G,True,None,500,375,53.9416,121.49,...,170.00,1,None,H,None,5.0,97.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...
501,997,"Road-750 Black, 44",BK-R19B-44,LB,True,Black,100,75,343.6496,539.99,...,19.77,4,R,L,U,2.0,31.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...
502,998,"Road-750 Black, 48",BK-R19B-48,LB,True,Black,100,75,343.6496,539.99,...,20.13,4,R,L,U,2.0,31.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...


In [88]:
dimensionProducto = dimensionProducto.merge(description, on='ProductModelID')


dimensionProducto.rename(columns={
    'Description' : 'EnglishDescription',
}, inplace=True)


dimensionProducto.drop(columns={
    'ProductDescriptionID'
}, inplace=True)

dimensionProducto

,ProductID,EnglishProductName,ProductNumber,WeightUnitMeasureCode,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,LargePhoto,EnglishDescription
0,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,1,R,H,U,14.0,6.0,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...,Our lightest and best quality aluminum frame m...
1,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,1,R,H,U,14.0,6.0,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...,لقد تم صناعة هيكل دراجتنا الألومنيوم الأخف وزن...
2,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,1,R,H,U,14.0,6.0,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...,Notre cadre en aluminium plus léger et de qual...
3,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,1,R,H,U,14.0,6.0,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...,เฟรมอลูมิเนียมคุณภาพสูงสุดและน้ำหนักเบาที่สุด ...
4,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,1,R,H,U,14.0,6.0,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...,מסגרת האלומיניום הקלה והאיכותית ביותר שלנו עשו...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1759,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,4,R,L,U,2.0,31.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...,إنها دراجة مناسبة للمبتدئين من البالغين؛ فهي ت...
1760,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,4,R,L,U,2.0,31.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...,Vélo d'adulte d'entrée de gamme ; permet une c...
1761,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,4,R,L,U,2.0,31.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...,จักรยานระดับเริ่มต้นสำหรับผู้ใหญ่ ให้ความสบายใ...
1762,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,4,R,L,U,2.0,31.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...,"אופני מבוגרים למתחילים; מציעים רכיבה נוחה ""מחו..."


In [89]:
dimensionProducto = dimensionProducto.merge(tablaProductModel, on='ProductModelID', how='left')
dimensionProducto.rename(columns={
    'Name' : 'ModelName',
}, inplace=True)

dimensionProducto


,ProductID,EnglishProductName,ProductNumber,WeightUnitMeasureCode,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,LargePhoto,EnglishDescription,ModelName
0,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,R,H,U,14.0,6.0,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...,Our lightest and best quality aluminum frame m...,HL Road Frame
1,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,R,H,U,14.0,6.0,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...,لقد تم صناعة هيكل دراجتنا الألومنيوم الأخف وزن...,HL Road Frame
2,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,R,H,U,14.0,6.0,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...,Notre cadre en aluminium plus léger et de qual...,HL Road Frame
3,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,R,H,U,14.0,6.0,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...,เฟรมอลูมิเนียมคุณภาพสูงสุดและน้ำหนักเบาที่สุด ...,HL Road Frame
4,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,R,H,U,14.0,6.0,2008-04-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x00\x00\x...,מסגרת האלומיניום הקלה והאיכותית ביותר שלנו עשו...,HL Road Frame
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1759,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,R,L,U,2.0,31.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...,إنها دراجة مناسبة للمبتدئين من البالغين؛ فهي ت...,Road-750
1760,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,R,L,U,2.0,31.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...,Vélo d'adulte d'entrée de gamme ; permet une c...,Road-750
1761,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,R,L,U,2.0,31.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...,จักรยานระดับเริ่มต้นสำหรับผู้ใหญ่ ให้ความสบายใ...,Road-750
1762,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,R,L,U,2.0,31.0,2013-05-30,NaT,b'GIF89a\xf0\x00\x95\x00\xf7\x00\x00\x8avw\xf7...,"אופני מבוגרים למתחילים; מציעים רכיבה נוחה ""מחו...",Road-750


In [90]:

# Crear columnas nuevas con None y truncar, cuando tengan data asignada luego será necesario
for col, length in {
    "SpanishProductName": 50,
    "FrenchProductName": 50,
    "FrenchDescription": 400,
    "ChineseDescription": 400,
    "ArabicDescription": 400,
    "HebrewDescription": 400,
    "ThaiDescription": 400,
    "GermanDescription": 400,
    "JapaneseDescription": 400,
    "TurkishDescription": 400,
    "SizeRange": 50,
    "DealerPrice": 50,
}.items():
    dimensionProducto[col] = None
    dimensionProducto[col] = dimensionProducto[col].str[:length]





dimensionProducto.loc[dimensionProducto["SellEndDate"].isnull(), "Status"] = "Current"
dimensionProducto.loc[dimensionProducto["SellEndDate"].notnull(), "Status"] = None

dimensionProducto

,ProductID,EnglishProductName,ProductNumber,WeightUnitMeasureCode,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,ChineseDescription,ArabicDescription,HebrewDescription,ThaiDescription,GermanDescription,JapaneseDescription,TurkishDescription,SizeRange,DealerPrice,Status
0,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,None,None,None,None,None,None,None,None,None,Current
1,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,None,None,None,None,None,None,None,None,None,Current
2,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,None,None,None,None,None,None,None,None,None,Current
3,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,None,None,None,None,None,None,None,None,None,Current
4,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,None,None,None,None,None,None,None,None,None,Current
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1759,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,None,None,None,None,None,None,None,None,None,Current
1760,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,None,None,None,None,None,None,None,None,None,Current
1761,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,None,None,None,None,None,None,None,None,None,Current
1762,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,None,None,None,None,None,None,None,None,None,Current


CAMBIO DE NOMBRES

In [91]:
dimensionProducto.rename(columns={
    'ProductID' : 'ProductKey',
    'ProductNumber': 'ProductAlternateKey',
    'SellStartDate' : 'StartDate',
    'SellEndDate' : 'EndDate',
    'ProductSubcategoryID' : 'ProductSubcategoryKey'
}, inplace=True)
dimensionProducto

,ProductKey,EnglishProductName,ProductAlternateKey,WeightUnitMeasureCode,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,ChineseDescription,ArabicDescription,HebrewDescription,ThaiDescription,GermanDescription,JapaneseDescription,TurkishDescription,SizeRange,DealerPrice,Status
0,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,None,None,None,None,None,None,None,None,None,Current
1,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,None,None,None,None,None,None,None,None,None,Current
2,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,None,None,None,None,None,None,None,None,None,Current
3,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,None,None,None,None,None,None,None,None,None,Current
4,680,"HL Road Frame - Black, 58",FR-R92B-58,LB,True,Black,500,375,1059.3100,1431.50,...,None,None,None,None,None,None,None,None,None,Current
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1759,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,None,None,None,None,None,None,None,None,None,Current
1760,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,None,None,None,None,None,None,None,None,None,Current
1761,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,None,None,None,None,None,None,None,None,None,Current
1762,999,"Road-750 Black, 52",BK-R19B-52,LB,True,Black,100,75,343.6496,539.99,...,None,None,None,None,None,None,None,None,None,Current


CARGAR A LA BODEGA

In [92]:
dimensionProducto.to_sql('dimensionProduct',motorBodegaDatos, if_exists='replace',index=False)

24